# Loading a Text Generation Model

**Step 1**: download a pre-trained model

* We rely on the library transformer


*   **AutoModelCausalLM** :  loads the pre-trained model weights for text generation

 Causal language models like AutoModelForCausalLM are commonly used for tasks such as text generation, language translation, and dialogue systems.

 The model is specifically tailored for “Causal” language modeling, which means it generates text in a unidirectional, left-to-right manner, predicting the next word in a sequence based on the preceding context.

*   **AutoTokenizer** : tokenizes the input text into numerical IDs that the model can process




In [1]:
import torch
from transformers import AutoModelForCausalLM,AutoTokenizer

We download the Phi-3-mini model

Architecture: Phi-3 Mini-4K-Instruct has 3.8B parameters and is a dense decoder-only Transformer model.

In [14]:
model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3-mini-4k-instruct",
                                             device_map="cuda",
                                             torch_dtype="auto",
                                             trust_remote_code=False
                                             )

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

We download the tokenizer associated to the model

In [15]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct",
                                          trust_remote_code=False,
                                          )

With transformers, we create

1. a pipeline for text-generation. This pipeline contains:

*   the model
*   the tokenizer
*   the output: the text generated

2. the configuration of some options for the tex generation


*   **return_full_text = false** : we only want the output of the model but not the prompt (the request)
*   **max_new_token** : the maximum number of tokens the model will generate

*   **do_sample** :  whether the model uses a sampling strategy to choose the next token. When we set **do_sample = False**, the model will select the most probable token


In [16]:
from transformers import pipeline, GenerationConfig

generation_config = GenerationConfig(max_new_tokens=100, do_sample=True)

generator = pipeline("text-generation",
                     model=model,
                     tokenizer=tokenizer,
                     return_full_text=False
                     )

The prompt is a dictionary with

"role" identifies who is speaking. Usual roles are:


*   "systems": instruction from the developer/operator
*   "users" : the human input
*   "assistant" : the model's response

"content": the actual text of the message






In [17]:
message = [
    {"role":"user","content":"create a funny joke about chikens."}
]

In [18]:
output=generator(message, generation_config=generation_config)

In [19]:
print(output[0]["generated_text"])

 Sure, here's a lighthearted chicken joke for you:

Why do chickens make terrible secret agents?

Because they always peck out the plans!


The transformers.pipeline transforms our message/prompt into a defined prompt template

In [20]:
prompt = generator.tokenizer.apply_chat_template(message, tokenize=False,add_generation_prompt=True)
print(prompt)

<|user|>
create a funny joke about chikens.<|end|>
<|assistant|>



# Controlling Model Output

Two parameters can be used to control the randomness of the output:



*   **temperature**

Control the randomness of the text generated

temperature = 0 : always generates the same response. The most likely word is always choosen

higher temperature : less probable words can be generated

*   **top_n**

A sampling method which controls which subset of tokens (nucleus) the LLM considers

LLM considers tokens untill it reaches their cumulative probability

top_n = 0.1 considers tokens until their cumulative probability is 10%

top_n= 1 considers all tokens



In [27]:
# setting temperature to its maximum level
generation_config = GenerationConfig(max_new_tokens=100, do_sample=True, temperature=1)


In [31]:
output=generator(prompt, generation_config=generation_config)

In [32]:
print(output[0]["generated_text"])

 Why did the chicken join a music band? Because it had the best "egg-sisting" beats!


In [33]:
# top_n
generation_config = GenerationConfig(max_new_tokens=100, do_sample=True, top_n=10)
output=generator(prompt, generation_config=generation_config)
print(output[0]["generated_text"])

 Why do chickens never play hide and seek?

Because good luck hiding when they always shout, "Cluck!" And they've got excellent eyesight to find you. They're not just good at roosting; they're excellent at unmasking you!


# Chain-of-Thought: Think before Answering

**One-shot prompt**
Prompting with a single example

In [61]:
one_shot_prompt=[
    {"role":"user","content": "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now? "},
    {"role": "assistant", "content": "The answer is 11"},
    {"role":"user","content":"The cafeteria had 23 apples. If they used 20 apples to make lunch and bought 6 more, how many apples do they have?"}
]

In [62]:
output=generator(one_shot_prompt, generation_config=generation_config)
print(output[0]["generated_text"])

 We begin by subtracting the 20 apples used from the initial 23, which leaves the cafeteria with 3 apples. Then, by adding the 6 new apples they bought, they have 3 + 6 = 9 apples remaining.


**Chain-of-thought prompt:**

We provide a reasoning example

In [63]:
cot_prompt=[
    {"role":"user","content": "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now? "},
    {"role": "assistant", "content": "Roger started with 5 balls. 2 cans of 3 balls each is 6 tennis balls. 5+6=11. The answer is 11"},
    {"role":"user","content":"The cafeteria had 23 apples. If they used 20 apples to make lunch and bought 6 more, how many apples do they have?"}
]

In [64]:
output=generator(cot_prompt, generation_config=generation_config)
print(output[0]["generated_text"])

 The cafeteria started with 23 apples. They used 20 apples for lunch, so we subtract those: 23 - 20 = 3 apples remaining. Then, they bought 6 more apples, so we add those: 3 + 6 = 9 apples. The cafeteria now has 9 apples.


**Zero-shot chain-of-thought**

We do not give an example but add "Let's think step by step"

alternative:

"Take a deep breath and think step by step"

"Lets's work through this problem step-by-step"

In [65]:
zeroshot_cot_prompt=[
    {"role":"user","content":"The cafeteria had 23 apples. If they used 20 apples to make lunch and bought 6 more, how many apples do they have? Let's think step by step"}
]

In [66]:
output=generator(zeroshot_cot_prompt, generation_config=generation_config)
print(output[0]["generated_text"])

 1. Start with the initial number of apples in the cafeteria: 23 apples.
2. Subtract the number of apples used to make lunch: 23 apples - 20 apples = 3 apples.
3. Add the number of apples bought: 3 apples + 6 apples = 9 apples.

The cafeteria has 9 apples now.


Tree-of-Thought:

idea: we ask the LLM to micmic the conversation between some experts.

Can be useful for creative tasks such as writing a story

In [67]:
zeroshot_tot_prompt =[
    {"role":"user","content":"Imagine three different experts are answering this question. All experts will write down 1 step of their thinking, then share it with the group.Then all expert will go on to the next step,etc. If any expert realizes they're wrong at any point then they leave. The question is 'The cafeteria had 23 apples. If they used 20 apples to make lunch and bought 6 more, how many apples do they have?'. Make sure to discuss the results."}

]

In [69]:
generation_config = GenerationConfig(max_new_tokens=500, do_sample=True, temperature=1)

In [70]:
output=generator(zeroshot_tot_prompt, generation_config=generation_config)
print(output[0]["generated_text"])

 Expert 1 (Step 1): 
The cafeteria started with 23 apples, used 20 for lunch, and then bought 6 more.

Expert 2 (Step 1): 
Begin with 23 apples in the cafeteria.

Expert 3 (Step 1): 
Starting number of apples is 23, which are essential to our calculation.

All Expert Results: So far, we agree that there were initially 23 apples in the cafeteria.

Expert 1 (Step 2):
After using 20 apples for lunch, the cafeteria would have -7 apples which is obviously incorrect.

Expert 2 (Step 2): 
After using 20 apples for lunch, you will subtract 20 from the initial 23, resulting in 3 apples left. Then, after buying 6 more, you add to the 3 apples that left, that means 9 apples.

Expert 3 (Step 2): 
The correct approach would involve subtracting the 20 apples used for lunch from the original 23, resulting in 3 apples. Then add the 6 apples bought, totaling 9 apples in the end.

Final Consensus: All experts agree that the cafeteria has 9 apples now, but it took a bit of explanation to reach this corre

**In-Context Learning**

We provide the LLM with an example of how we want him to behave

{"role": "system", "content": "You are a helpful AI assistant."} = we instruct the LLM how to behave

second + third messages : example of what we expect from the LLM

final request: what we ask

In [43]:
messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": "Can you provide ways to eat combinations of bananas and dragonfruits?"},
    {"role": "assistant", "content": "Sure! Here are some ways to eat bananas and dragonfruits together: 1. Banana and dragonfruit smoothie: Blend bananas and dragonfruits together with some milk and honey. 2. Banana and dragonfruit salad: Mix sliced bananas and dragonfruits together with some lemon juice and honey."},
    {"role": "user", "content": "What about solving an 2x + 3 = 7 equation?"},
]

In [44]:
output=generator(messages, generation_config=generation_config)

In [45]:
print(output[0]['generated_text'])

 To solve the equation "2x + 3 = 7," follow these steps:

1. Subtract 3 from both sides of the equation: 2x + 3 - 3 = 7 - 3 This simplifies to 2x = 4.
2. Divide both sides by 2: 2x / 2 = 4 / 2 This simplifies to x = 2.

So, the solution to this


Another example without the pipeline

In [37]:
prompt ="write an email apologizing to Mister Smith for not attending his anniversary. Explain why I could not be present.<|assistant|>"

In [38]:
encoded_input = tokenizer(prompt, return_tensors="pt")
input_ids = encoded_input.input_ids.to("cuda")
attention_mask = encoded_input.attention_mask.to("cuda")

In [39]:
input_ids

tensor([[ 2436,   385,  4876, 27746,  5281,   304,   341,  1531,  7075,   363,
           451,  1098,  2548,   670,  6957, 27547, 29889, 12027,  7420,  2020,
           306,  1033,   451,   367,  2198, 29889, 32001]], device='cuda:0')

In [40]:
generation_output = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=100)

In [41]:
print(tokenizer.decode(generation_output[0]))

write an email apologizing to Mister Smith for not attending his anniversary. Explain why I could not be present.<|assistant|> Subject: Heartfelt Apologies for Missing Mr. Smith's Anniversary


Dear Mr. Smith,


I hope this message finds you in good spirits. I am writing to express my sincerest apologies for not being able to attend your anniversary celebration. Unfortunately, an unforeseen medical emergency arose that required my immediate attention, and I was unable to inform you in time.


Please know that my absence was
